# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook explores the FAIR^2 dataset using the `mlcroissant` library, providing examples of loading, inspecting, and analyzing data defined via a Croissant schema.

### Dataset Source
The dataset source is provided via a Croissant schema URL and uses consistent reference to dataset entities via their `@id` fields.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
# Access the JSON metadata
metadata_json = dataset.metadata.to_json()

print(f"Dataset name: {metadata_json['name']}")
print(f"Description: {metadata_json['description']}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

Below, we list the dataset's record sets. For each, we print its fields and columns, referencing all by `@id`.

In [ ]:
# Retrieve available record sets by their @id
record_sets = dataset.record_sets

if not record_sets:
    # The dataset may have a single main record set implicitly as per Croissant 1.0 spec
    print("No explicit record sets defined, defaulting to schema main table.")
    # Try extracting possible record set info from metadata
    # (as mlcroissant exposes via dataset.record_sets)
    record_sets = dataset.record_sets  # might be empty; mlcroissant may still allow records(None)
else:
    print(f"Found {len(record_sets)} record sets.")

# Print each record set's @id, label, and associated field @ids
for idx, rs in enumerate(record_sets):
    print(f"[{idx}] RecordSet @id: {rs['@id']}")
    label = rs.get('name', rs.get('label', None))
    if label:
        print(f"      name: {label}")
    fields = rs.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]
    print("      Fields & Columns @ids:")
    for f in fields:
        if isinstance(f, dict):
            print(f"        - Field @id: {f.get('@id', '<missing>')}")
        elif isinstance(f, str):
            print(f"        - Field @id: {f}")
    print("----------------------")

# If no explicit record sets, demo listing first 3 records
if not record_sets:
    # Use dataset.records() with default/main record set
    print("First 3 records (default/main record set):")
    for i, rec in enumerate(dataset.records()):
        print(json.dumps(rec, indent=2))
        if i >= 2:
            break

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis.

We'll extract data by a record set `@id`. If no explicit record sets exist, we'll use the main table (`record_set=None`).

In [ ]:
# Assemble record set IDs for extraction
all_recordset_ids = [rs['@id'] for rs in dataset.record_sets] if dataset.record_sets else [None]
dataframes = {}

for record_set_id in all_recordset_ids:
    recs = list(dataset.records(record_set=record_set_id))
    # Convert to DataFrame
    if recs:
        df = pd.DataFrame(recs)
        dataframes[record_set_id or 'main'] = df
        print(f"Loaded {len(df)} records for record set: {record_set_id or 'main'}")
        print(f"Fields: {df.columns.tolist()}")

# For this FAIR^2 dataset, the main record set is most likely in the main table
main_record_set_id = all_recordset_ids[0]
print(dataframes[main_record_set_id or 'main'].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, demonstrating filtering, normalization, and grouping/categorization. As the dataset contains protein mass spectrometry, suitable fields include e.g. `'MW [kDa]'` for molecular weight analysis.

The following EDA steps reference all columns by their `@id` (i.e., JSON key names from the DataFrame columns).

In [ ]:
# Select DataFrame for main record set
df = dataframes[main_record_set_id or 'main']
# List numeric-looking fields
numeric_candidates = [col for col in df.columns if df[col].dtype.kind in 'fi']
print("Numeric field candidates for EDA:", numeric_candidates)

# If known field, e.g. 'MW [kDa]', else pick the first numeric column
# Use the column @id (from DataFrame)
if 'MW [kDa]' in df.columns:
    numeric_field = 'MW [kDa]'
else:
    # Fallback to first numeric
    numeric_field = numeric_candidates[0]

print(f"Using numeric field @id: {numeric_field}")

# EDA step 1: Filter for high molecular weight proteins (> 60 kDa as example)
threshold = 60
filtered_df = df[df[numeric_field].astype(float) > threshold].copy()

print(f"Filtered records with {numeric_field} > {threshold}:")
print(filtered_df.head())

# EDA step 2: Normalize selected numeric field
filtered_df[numeric_field + '_normalized'] = (
    filtered_df[numeric_field].astype(float) - filtered_df[numeric_field].astype(float).mean()
) / filtered_df[numeric_field].astype(float).std()

print(f"Normalized {numeric_field} for filtered records:")
print(filtered_df[[numeric_field, numeric_field + '_normalized']].head())

# EDA step 3: Group by another available field. Use e.g. 'AC' if present (protein accession), else skip.
group_field_candidates = [col for col in df.columns if col != numeric_field]
group_field = None
for candidate in ['Description', 'AC', 'Gene Name']:
    if candidate in df.columns:
        group_field = candidate
        break
if group_field:
    grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True).sort_values(by=numeric_field, ascending=False)
    print(f"Grouped data by {group_field}:")
    print(grouped_df[[numeric_field, numeric_field + '_normalized']].head())

## 5. Visualization
Visualize the distribution of molecular weights using a histogram, and protein abundances (if available).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot histogram of molecular weights
plt.figure(figsize=(8, 4))
sns.histplot(df[numeric_field].astype(float), bins=30, kde=True)
plt.xlabel(numeric_field)
plt.ylabel('Frequency')
plt.title('Distribution of Protein Molecular Weights')
plt.tight_layout()
plt.show()

# If another numeric field (e.g. abundance) present, plot scatter
possible_abundance_cols = [col for col in df.columns if 'abundance' in col.lower() or 'norm' in col.lower()]
if possible_abundance_cols:
    abundance_field = possible_abundance_cols[0]
    plt.figure(figsize=(7, 5))
    plt.scatter(
        df[numeric_field].astype(float),
        df[abundance_field].astype(float),
        alpha=0.6
    )
    plt.xlabel(numeric_field)
    plt.ylabel(abundance_field)
    plt.title('Protein Molecular Weight vs. Abundance')
    plt.tight_layout()
    plt.show()

## 6. Conclusion
We have illustrated how to load, filter, normalize, and visualize records from the FAIR^2 mass spectrometry dataset using `mlcroissant`—with all references to data entities by their `@id` per schema best practices. The approach generalizes to any Croissant-compliant package for Findable, Accessible, Interoperable, Reusable FAIR-by-design data.

**Key points:**
- All data and processing steps consistently reference `@id`s for schema entities.
- We demonstrated numeric filtering, normalization, grouping, and visualization for protein data.
- The mlcroissant library enables robust, semantically referenced FAIR data exploration for science and ML.